In [71]:
import torch 


In [73]:
torch.__version__

'2.13.0+cu130'

In [76]:
from torch import nn
from torch.optim import Adam
from torch.nn import CrossEntropyLoss
from torch.utils.data import DataLoader

import torchvision
from torchvision.transforms import transforms
from torchvision.datasets import ImageFolder

In [77]:
training_trans=transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomRotation(15),
    transforms.ToTensor()
])
testing_trans=transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor()
])

In [78]:
training_trans

Compose(
    Resize(size=(224, 224), interpolation=bilinear, max_size=None, antialias=True)
    RandomRotation(degrees=[-15.0, 15.0], interpolation=nearest, expand=False, fill=0)
    ToTensor()
)

In [79]:
training_data=ImageFolder(root=r"E:\Datasets\plant_classification\train\train",transform=training_trans)
testing_data=ImageFolder(root=r"E:\Datasets\plant_classification\test\test",transform=testing_trans)

In [80]:
im,l=training_data[0]


In [81]:
train_data_loader=DataLoader(dataset=training_data,batch_size=64,shuffle=True)
test_data_loader=DataLoader(dataset=testing_data,batch_size=64,shuffle=True)

In [82]:
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
device 

device(type='cuda')

In [83]:
classes=training_data.classes
len(classes)

12

In [84]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1=nn.Conv2d(in_channels=3,out_channels=12,kernel_size=3)
        self.conv2=nn.Conv2d(in_channels=12,out_channels=24,kernel_size=3)
        self.conv3=nn.Conv2d(in_channels=24,out_channels=28,kernel_size=3)
        self.pool=nn.MaxPool2d(kernel_size=3,stride=3)
        self.relu=nn.ReLU(inplace=True)

        self.fc1=nn.Linear(in_features=28*7*7,out_features=1513)
        self.fc2=nn.Linear(in_features=1513,out_features=800)
        self.fc3=nn.Linear(in_features=800,out_features=400)
        self.fc4=nn.Linear(in_features=400,out_features=12)
    def forward(self,x):
        x=self.pool(self.relu(self.conv1(x)))
        x=self.pool(self.relu(self.conv2(x)))
        x=self.pool(self.relu(self.conv3(x)))
        x=torch.flatten(x,1)
        return (self.fc4(self.relu(self.fc3(self.relu(self.fc2(self.relu(self.fc1(x))))))))
        

In [86]:
model=CNN().to(device=device)
model

AcceleratorError: CUDA error: device-side assert triggered
Search for `cudaErrorAssert' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1


In [53]:
optimizer=Adam(params=model.parameters(),lr=0.001)
loss_func=CrossEntropyLoss()

In [54]:
model.train()
for j in range(20):
    for i,(img,label) in enumerate(train_data_loader):
        img,label=img.to(device),label.to(device)
        y_pred=model(img)
        loss=loss_func(y_pred,label)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        print(f"loss for {i} is {loss.item()}")

loss for 0 is 2.479769468307495
loss for 1 is 2.473191976547241
loss for 2 is 2.451883316040039
loss for 3 is 2.4183692932128906
loss for 4 is 2.4736390113830566
loss for 5 is 2.3734703063964844
loss for 6 is 2.2354588508605957
loss for 7 is 2.4310872554779053
loss for 8 is 2.4594016075134277
loss for 9 is 2.294008255004883
loss for 10 is 2.376101016998291
loss for 11 is 2.2760205268859863
loss for 12 is 2.3740577697753906
loss for 13 is 2.392040252685547
loss for 14 is 2.3765785694122314
loss for 15 is 2.3601877689361572
loss for 16 is 2.327148199081421
loss for 17 is 2.330610752105713
loss for 18 is 2.30424165725708
loss for 19 is 2.32332444190979
loss for 20 is 2.2015440464019775
loss for 21 is 2.0836238861083984
loss for 22 is 2.3331165313720703
loss for 23 is 2.133364200592041
loss for 24 is 2.1983814239501953
loss for 25 is 2.166867256164551
loss for 26 is 2.2195377349853516
loss for 27 is 1.9775333404541016
loss for 28 is 2.038411855697632
loss for 29 is 2.025313377380371
loss f

In [74]:
model.eval()
correct = 0
total = len(test_data_loader.dataset)

with torch.no_grad():
    for images, labels in test_data_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        preds = torch.argmax(outputs, dim=1)
        correct += (preds == labels).sum().item()

accuracy = correct / total
print(f"Accuracy: {accuracy:.4f}")


AcceleratorError: CUDA error: device-side assert triggered
Search for `cudaErrorAssert' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1


In [75]:
from PIL import Image



# 2. Load image from path
img_path = r"C:\Users\user\Downloads\download.jpg"
image = Image.open(img_path).convert("RGB")

# 3. Apply transform
image = testing_trans(image)

# 4. Add batch dimension
image = image.unsqueeze(0).to(device)

# 5. Run inference
model.eval()
with torch.no_grad():
    outputs = model(image)
    preds = torch.argmax(outputs, dim=1)

print("Predicted class index:", classes[preds.item()])
outputs


AcceleratorError: CUDA error: device-side assert triggered
Search for `cudaErrorAssert' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
